In [ ]:
pip install tflite-runtime


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 20.7 MB/s eta 0:00:00


In [ ]:
# Load your TensorFlow Lite model
interpreter = tf.lite.Interpreter(model_path='/content/conv_lstm_model.tflite')
interpreter.allocate_tensors()

# Get input and output tensor details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Check the shape of the input required by the model
input_shape = input_details[0]['shape']
print(f'Input shape required by the model: {input_shape}')


Input shape required by the model: [1 4 1 5 6]


In [ ]:
import pandas as pd

# Load the dataset (replace with the correct path to your dataset)
data = pd.read_csv('/content/test_motion_data.csv')

# Show the first few rows of the dataset
print(data.head())

# Check the overall structure and size of the dataset
print(f'Dataset shape: {data.shape}')


       AccX      AccY      AccZ     GyroX     GyroY     GyroZ       Class  \
0  0.758194 -0.217791  0.457263  0.000000  0.000000  0.000000  AGGRESSIVE   
1  0.667560 -0.038610  0.231416 -0.054367 -0.007712  0.225257  AGGRESSIVE   
2  2.724449 -7.584121  2.390926  0.023824  0.013668 -0.038026  AGGRESSIVE   
3  2.330950 -7.621754  2.529024  0.056810 -0.180587 -0.052076  AGGRESSIVE   
4  2.847215 -6.755621  2.224640 -0.031765 -0.035201  0.035277  AGGRESSIVE   

   Timestamp  
0     818922  
1     818923  
2     818923  
3     818924  
4     818924  
Dataset shape: (3084, 8)


In [ ]:
import numpy as np
import tensorflow as tf

# Load the TensorFlow Lite model
def load_tflite_model(model_path):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    return interpreter

# Prepare the test data
def prepare_test_data():
    # Manually define the data for testing with the required dimensions
    # Shape: (1 sample, 4 columns, 1 row, 5 timesteps, 6 features)

    small_data = np.array([[[[0.5, -0.2, 0.4, 0.1, -0.3, 0.2],   # Column 1, timestep 1
                            [0.6, -0.1, 0.3, 0.2, -0.4, 0.3],   # Column 1, timestep 2
                            [0.7, -0.2, 0.2, 0.1, -0.3, 0.1],   # Column 1, timestep 3
                            [0.8, -0.3, 0.4, 0.3, -0.2, 0.2],   # Column 1, timestep 4
                            [0.9, -0.2, 0.3, 0.1, -0.4, 0.3]],  # Column 1, timestep 5

                           [[0.4, -0.1, 0.5, 0.0, -0.2, 0.1],   # Column 2, timestep 1
                            [0.5, -0.2, 0.4, 0.1, -0.3, 0.2],   # Column 2, timestep 2
                            [0.6, -0.1, 0.3, 0.2, -0.4, 0.3],   # Column 2, timestep 3
                            [0.7, -0.2, 0.2, 0.1, -0.3, 0.1],   # Column 2, timestep 4
                            [0.8, -0.3, 0.4, 0.3, -0.2, 0.2]],  # Column 2, timestep 5

                           [[0.4, -0.1, 0.5, 0.0, -0.2, 0.1],   # Column 3, timestep 1
                            [0.5, -0.2, 0.4, 0.1, -0.3, 0.2],   # Column 3, timestep 2
                            [0.6, -0.1, 0.3, 0.2, -0.4, 0.3],   # Column 3, timestep 3
                            [0.7, -0.2, 0.2, 0.1, -0.3, 0.1],   # Column 3, timestep 4
                            [0.8, -0.3, 0.4, 0.3, -0.2, 0.2]],  # Column 3, timestep 5

                           [[0.4, -0.1, 0.5, 0.0, -0.2, 0.1],   # Column 4, timestep 1
                            [0.5, -0.2, 0.4, 0.1, -0.3, 0.2],   # Column 4, timestep 2
                            [0.6, -0.1, 0.3, 0.2, -0.4, 0.3],   # Column 4, timestep 3
                            [0.7, -0.2, 0.2, 0.1, -0.3, 0.1],   # Column 4, timestep 4
                            [0.8, -0.3, 0.4, 0.3, -0.2, 0.2]]   # Column 4, timestep 5
                         ]])

    # Add an extra dimension to represent the row (N_ROWS = 1)
    small_data = np.expand_dims(small_data, axis=2)  # Shape now becomes (1, 4, 1, 5, 6)

    return small_data

# Run the model on the prepared data
def run_tflite_model(interpreter, input_data):
    # Get input and output tensors
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # Set the input tensor
    input_shape = input_details[0]['shape']
    input_data = np.array(input_data, dtype=np.float32)
    interpreter.set_tensor(input_details[0]['index'], input_data)

    # Invoke the model
    interpreter.invoke()

    # Get the output tensor
    output_data = interpreter.get_tensor(output_details[0]['index'])
    return output_data

# Main function to load the model and test it
def main():
    model_path = '/content/conv_lstm_model.tflite'  # Path to your .tflite model
    interpreter = load_tflite_model(model_path)

    # Prepare the test data
    test_data = prepare_test_data()

    # Run the model on the test data
    result = run_tflite_model(interpreter, test_data)

    # Output the result
    print(f"Model output: {result}")

if __name__ == "__main__":
    main()


Model output: [[0.87782997 0.12217001]]


In [ ]:
import time

# Start timer
start_time = time.time()

# Run inference
interpreter.invoke()

# Measure elapsed time
inference_time = time.time() - start_time
print(f"Inference Time: {inference_time} seconds")

Inference Time: 0.0014865398406982422 seconds
